In [1]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT=Path('/tmp/e07_source'); ORCH=Path('/tmp/e07_orchestration')
OUT=Path('/kaggle/working')
SOURCE='5bb03b18943e7c45a305ce1d78b3446c31829ae7'
ORCH_COMMIT='5db179ce6621acc199701b0327ce8d08d9eeb19c'
env={**os.environ,'UV_PYTHON_INSTALL_DIR':'/tmp/e07_python','UV_CACHE_DIR':'/tmp/e07_cache','UV_NATIVE_TLS':'true','CUBLAS_WORKSPACE_CONFIG':':4096:8','OMP_NUM_THREADS':'1','MPLCONFIGDIR':'/tmp/e07_mpl','MPLBACKEND':'Agg','PYTHONPATH':str(ORCH)}
def command(args,cwd=None):
    print('EXEC',args,flush=True)
    with (OUT/'bootstrap.log').open('a') as log:
        result=subprocess.run(args,cwd=cwd,env=env,stdout=log,stderr=subprocess.STDOUT)
    if result.returncode:
        print((OUT/'bootstrap.log').read_text()[-4000:],flush=True)
        raise RuntimeError('Bootstrap or qualification failed; preserve output')
assert not ROOT.exists() and not ORCH.exists()
for destination,commit in [(ROOT,SOURCE),(ORCH,ORCH_COMMIT)]:
    command(['git','clone','https://github.com/rayyan-merchant/Nano-Language-Model.git',str(destination)])
    command(['git','checkout','--detach',commit],destination)
    assert not subprocess.check_output(['git','status','--porcelain'],cwd=destination,text=True).strip()
command([sys.executable,'-m','pip','install','--target','/tmp/e07_tools','uv==0.12.7'])
UV=Path('/tmp/e07_tools/bin/uv')
command([str(UV),'python','install','3.11.14'])
command([str(UV),'venv','--seed','--python','3.11.14','/tmp/e07_venv'])
PY=Path('/tmp/e07_venv/bin/python')
freeze=(ROOT/'docs/qualifications/evidence/rayyan_requirements_freeze.txt').read_text().splitlines()
lock=OUT/'qualified-lock.txt';lock.write_text(chr(10).join(x for x in freeze if not x.startswith('-e '))+chr(10))
command([str(UV),'pip','install','--python',str(PY),'torch==2.5.1+cu124','--index-url','https://download.pytorch.org/whl/cu124'])
command([str(UV),'pip','install','--python',str(PY),'-r',str(lock)])
command([str(UV),'pip','install','--python',str(PY),'-e',str(ROOT),'--no-deps'])
command([str(PY),'-m','pip','check'])
archives=list(Path('/kaggle/input').rglob('E07_smoke_bundle.zip'))
assert len(archives)==1
ARCHIVE=archives[0]; MANIFEST=ARCHIVE.parent/'E07_smoke_manifest.json'
import hashlib
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()=='7f0dbb4a5522b9ed5b8b80fffb97f079cedadc80126256e47fb8e85962eec50e'
assert json.loads(MANIFEST.read_text())['archive_sha256']=='7f0dbb4a5522b9ed5b8b80fffb97f079cedadc80126256e47fb8e85962eec50e'
RECOVERY_OUT=OUT/'smoke_persistent_recovery'
command([str(PY),str(ORCH/'scripts/pilot/e07_kaggle.py'),'recover-smoke','--root',str(ROOT),'--output',str(RECOVERY_OUT),'--archive',str(ARCHIVE),'--manifest',str(MANIFEST)],ROOT)
print((RECOVERY_OUT/'persistent_recovery.json').read_text(),flush=True)
RESEARCH_OUT=OUT/'e07_pilot'; STORE=OUT/'e07_store'
print('FRESH COMPATIBLE CUDA RECOVERY PASSED; RESEARCH NOT YET STARTED',flush=True)


EXEC ['git', 'clone', 'https://github.com/rayyan-merchant/Nano-Language-Model.git', '/tmp/e07_source']
EXEC ['git', 'checkout', '--detach', '5bb03b18943e7c45a305ce1d78b3446c31829ae7']
EXEC ['git', 'clone', 'https://github.com/rayyan-merchant/Nano-Language-Model.git', '/tmp/e07_orchestration']
EXEC ['git', 'checkout', '--detach', '5db179ce6621acc199701b0327ce8d08d9eeb19c']
EXEC ['/usr/bin/python3', '-m', 'pip', 'install', '--target', '/tmp/e07_tools', 'uv==0.12.7']
EXEC ['/tmp/e07_tools/bin/uv', 'python', 'install', '3.11.14']
EXEC ['/tmp/e07_tools/bin/uv', 'venv', '--seed', '--python', '3.11.14', '/tmp/e07_venv']
EXEC ['/tmp/e07_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e07_venv/bin/python', 'torch==2.5.1+cu124', '--index-url', 'https://download.pytorch.org/whl/cu124']
EXEC ['/tmp/e07_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e07_venv/bin/python', '-r', '/kaggle/working/qualified-lock.txt']
EXEC ['/tmp/e07_tools/bin/uv', 'pip', 'install', '--python', '/tmp/e07_venv/bi

In [2]:
assert json.loads((RECOVERY_OUT/'persistent_recovery.json').read_text())['status']=='saved_output_archive_and_fresh_compatible_CUDA_full_state_recovery_passed'
assert not RESEARCH_OUT.exists() and not STORE.exists()
research_log=(OUT/'E07_research.log').open('x')
research_started=time.time()
research_proc=subprocess.Popen([str(PY),str(ORCH/'scripts/pilot/e07_kaggle.py'),'research','--root',str(ROOT),'--output',str(RESEARCH_OUT),'--store',str(STORE),'--recovery-receipt',str(RECOVERY_OUT/'persistent_recovery.json')],cwd=ROOT,env=env,stdout=research_log,stderr=subprocess.STDOUT,start_new_session=True)
print('E07 seed42 research PID',research_proc.pid,'authorized exact stop 5000 updates',flush=True)


E07 seed42 research PID 281 authorized exact stop 5000 updates


In [5]:
runs=list((RESEARCH_OUT/'runs').glob('*')) if (RESEARCH_OUT/'runs').exists() else []
run_dir=runs[0] if len(runs)==1 else None
events=[json.loads(x) for x in (run_dir/'events.jsonl').read_text().splitlines()] if run_dir and (run_dir/'events.jsonl').exists() else []
metrics=[e['metric'] for e in events if e.get('event')=='step']
validations=[r for r in metrics if 'validation' in r]
snaps=[p for p in (STORE/'research_snapshots').glob('*') if (p/'export_manifest.json').exists()]
print(json.dumps({'returncode':research_proc.poll(),'run_id':run_dir.name if run_dir else None,'completed_updates':metrics[-1]['step'] if metrics else 0,'full_validations':len(validations),'latest_validation':validations[-1] if validations else None,'committed_snapshots':len(snaps),'failures':[e for e in events if e.get('event')=='failure'],'log_tail':(OUT/'E07_research.log').read_text()[-500:]},indent=2))


{
  "returncode": null,
  "run_id": "E07_seed42_f00363fe26f64ec1b3bb5ad00d19c4bd",
  "completed_updates": 2249,
  "full_validations": 8,
  "latest_validation": {
    "step": 2000,
    "training_loss": 1.7286601513624191,
    "training_tokens": 8192000,
    "training_bytes": 8192000,
    "unique_target_bytes": 1003559,
    "train_seconds": 371.7766485390006,
    "tokens_per_second": 22034.735188970946,
    "steps_per_second": 5.37957402074486,
    "peak_gpu_bytes": 169584128,
    "gradient_norm": 0.497603178024292,
    "amp_skipped_updates": 0,
    "lr": 0.0003,
    "accumulation": 8,
    "validation": {
      "loss": 1.7639447682339413,
      "perplexity": 5.835411396562859,
      "bpb": 2.544834369533139,
      "nll_sum": 97923.62986373901,
      "scored_tokens": 55514,
      "scored_bytes": 55514,
      "conditioning_bytes": 256,
      "evaluated_text_bytes": 55770,
      "mode": "sliding",
      "context_length": 128
    },
    "generalization_gap": 0.03528461687152218
  },
  "commi